In [1]:
# ==========================================
# CELDA 1: Configuración y Carga de Infraestructura
# ==========================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from ortools.sat.python import cp_model
import warnings
warnings.filterwarnings('ignore')

# 1. Conectar a la BD (Ajusta tus credenciales si es necesario)
user = "Joasro"
password = "Akriila123." 
host = "localhost"
db = "dss_academico_unah"

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

print("Cargando logística desde la BD...")

# 🛑 MAGIA: Solo traemos a los docentes habilitados para este periodo
df_docentes = pd.read_sql("SELECT * FROM docentes_activos WHERE Disponible = 1", engine)

df_docente_area = pd.read_sql("SELECT * FROM docente_area", engine)
df_aulas = pd.read_sql("SELECT * FROM espacios_fisicos", engine)

df_malla = pd.read_sql("""
    SELECT ID_Clase, Nombre_Clase, Codigo_Oficial, ID_Area, 
           Requiere_Laboratorio, Permite_Presencial, Permite_Virtual 
    FROM malla_curricular
""", engine)

df_demanda = pd.read_csv('../data/demanda_proyectada_2026.csv')
if 'ID_Clase' not in df_demanda.columns:
    df_demanda = pd.merge(df_demanda, df_malla[['Codigo_Oficial', 'ID_Clase']], on='Codigo_Oficial', how='left')

HORARIOS = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] 
CAPACIDAD_ESTANDAR = 25 

print(f"✅ Datos listos: {len(df_docentes)} docentes DISPONIBLES y {len(df_aulas)} aulas físicas.")

Cargando logística desde la BD...
✅ Datos listos: 8 docentes DISPONIBLES y 8 aulas físicas.


In [2]:
# ==========================================
# CELDA 2: Generador de Variables (Patrones a Prueba de Balas)
# ==========================================
import itertools
model = cp_model.CpModel()
secciones_posibles = {}
duraciones = {} 
ID_AULA_VIRTUAL = 999 

def extraer_hora_entera(val):
    if pd.isnull(val): return None
    if hasattr(val, 'seconds'): return val.seconds // 3600
    try: return int(str(val).split(':')[0])
    except: return None

def obtener_lista_bloqueos(val):
    if pd.isnull(val) or str(val).strip() == "": return []
    try: return [int(x.strip()) for x in str(val).split(',') if x.strip().isdigit()]
    except: return []

def generar_patrones_dias(uv, dias_trabajo_docente):
    dias_profe = [d.strip() for d in str(dias_trabajo_docente).split(',')]
    patrones_validos = []
    
    if len(dias_profe) <= 2:
        for d in dias_profe: patrones_validos.append(d)
        return patrones_validos

    # Patrones ideales de la UNAH
    if uv >= 5 and all(dia in dias_profe for dia in ['Lu','Ma','Mi','Ju','Vi']):
        patrones_validos.append('Lu,Ma,Mi,Ju,Vi')
    elif uv == 4 and all(dia in dias_profe for dia in ['Lu','Ma','Mi','Ju']):
        patrones_validos.append('Lu,Ma,Mi,Ju') 
    elif uv <= 3:
        if all(dia in dias_profe for dia in ['Lu','Mi','Vi']): patrones_validos.append('Lu,Mi,Vi')
        if all(dia in dias_profe for dia in ['Ma','Ju']): patrones_validos.append('Ma,Ju')
            
    # 🛑 FALLBACK A PRUEBA DE BALAS: 
    # Si el profe tiene días salteados (ej. Lu, Mi, Ju, Vi), 
    # la IA armará una combinación con los días que sí tiene.
    if not patrones_validos and len(dias_profe) >= uv:
        comb = list(itertools.combinations(dias_profe, uv))[0]
        patrones_validos.append(",".join(comb))
                
    return patrones_validos

print("Generando combinaciones de horarios y aulas...")

for idx, row in df_demanda.iterrows():
    id_clase = row['ID_Clase']
    num_s = int(np.ceil(row['Cupos_Estimados'] / CAPACIDAD_ESTANDAR))
    
    info_c = df_malla[df_malla['ID_Clase'] == id_clase].iloc[0]
    uv = int(info_c.get('Unidades_Valorativas', 4))
    area_clase, p_presencial, p_virtual = info_c['ID_Area'], info_c['Permite_Presencial'], info_c['Permite_Virtual']

    for s in range(num_s):
        for d_idx, doc in df_docentes.iterrows():
            id_doc, tipo_d = doc['ID_Docente'], doc['Tipo_Docente']
            
            especialidades = df_docente_area[df_docente_area['ID_Docente'] == id_doc]['ID_Area'].values
            if area_clase not in especialidades: continue

            patrones_dias = generar_patrones_dias(uv, doc['Dias_Trabajo'])
            if not patrones_dias: continue 

            h_inicio = extraer_hora_entera(doc['Hora_Inicio_Turno'])
            h_fin = extraer_hora_entera(doc['Hora_Fin_Turno'])
            bloqueos = obtener_lista_bloqueos(doc['Horas_Bloqueadas']) 

            for patron in patrones_dias:
                duracion = uv if len(patron.split(',')) == 1 else 1
                
                for h in HORARIOS:
                    if h_inicio is not None and h < h_inicio: continue
                    if h_fin is not None and (h + duracion) > h_fin: continue
                    if any(hora_clase in bloqueos for hora_clase in range(h, h + duracion)): continue 

                    if tipo_d == 'Tegucigalpa' and p_virtual == 1:
                        v_key = (id_clase, s, id_doc, patron, h, ID_AULA_VIRTUAL, 'Virtual')
                        secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_doc}_p{patron}_h{h}_V")
                        duraciones[v_key] = duracion
                    else:
                        if p_virtual == 1:
                            v_key = (id_clase, s, id_doc, patron, h, ID_AULA_VIRTUAL, 'Virtual')
                            secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_doc}_p{patron}_h{h}_V")
                            duraciones[v_key] = duracion
                        if p_presencial == 1:
                            for a_idx, aula in df_aulas.iterrows():
                                v_key = (id_clase, s, id_doc, patron, h, aula['ID_Espacio'], 'Presencial')
                                secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_doc}_p{patron}_h{h}_a{aula['ID_Espacio']}")
                                duraciones[v_key] = duracion

print(f"🧩 Variables flexibles generadas: {len(secciones_posibles)}")

Generando combinaciones de horarios y aulas...
🧩 Variables flexibles generadas: 5195


In [3]:
# ==========================================
# CELDA 3: Restricciones de Carga y Choques Reales
# ==========================================
dias_semana_evaluar = ['Lu', 'Ma', 'Mi', 'Ju', 'Vi', 'Sa', 'Do']

model.Add(sum(secciones_posibles.values()) >= 21)
model.Add(sum(secciones_posibles.values()) <= 28)

# 1. Choques de Aula (Evaluando la DURACIÓN real para que no quede libre a medias)
for id_a in df_aulas['ID_Espacio']:
    for h_eval in HORARIOS:
        for dia in dias_semana_evaluar:
            v_a = []
            for v_key, var in secciones_posibles.items():
                c, s, d, p, h_inicio, aula, m = v_key
                if aula == id_a and dia in p.split(',') and m != 'Virtual':
                    # Magia: Si la hora evaluada está dentro del rango que dura la clase, la bloquea
                    if h_inicio <= h_eval < (h_inicio + duraciones[v_key]):
                        v_a.append(var)
            if v_a: model.Add(sum(v_a) <= 1)

# 2. Choques de Docente y Carga Máxima (Igual, respetando su duración)
docentes_activos_vars = {}
for id_d in df_docentes['ID_Docente']:
    for h_eval in HORARIOS:
        for dia in dias_semana_evaluar:
            vars_doc = []
            for v_key, var in secciones_posibles.items():
                c, s, d, p, h_inicio, aula, m = v_key
                if d == id_d and dia in p.split(','):
                    if h_inicio <= h_eval < (h_inicio + duraciones[v_key]):
                        vars_doc.append(var)
            if vars_doc: model.Add(sum(vars_doc) <= 1)
    
    clases_profe = [var for v_key, var in secciones_posibles.items() if v_key[2] == id_d]
    if clases_profe:
        model.Add(sum(clases_profe) <= 4) 
        es_usado = model.NewBoolVar(f'u_{id_d}')
        model.Add(sum(clases_profe) > 0).OnlyEnforceIf(es_usado)
        model.Add(sum(clases_profe) == 0).OnlyEnforceIf(es_usado.Not())
        docentes_activos_vars[id_d] = es_usado

# 3. Restricción de Sección Única
for idx, row in df_demanda.iterrows():
    id_c = row['ID_Clase']
    num_s = int(np.ceil(row['Cupos_Estimados'] / CAPACIDAD_ESTANDAR))
    for sec_idx in range(num_s):
        ops = [var for v_key, var in secciones_posibles.items() if v_key[0] == id_c and v_key[1] == sec_idx]
        if ops: model.Add(sum(ops) <= 1)

# 4. Límites Presupuestarios (2 Emergentes, 2 TGU)
def contar_contratos(tipo):
    ids = df_docentes[df_docentes['Tipo_Docente'] == tipo]['ID_Docente'].values
    return [docentes_activos_vars[i] for i in ids if i in docentes_activos_vars]

emergentes, tgu = contar_contratos('Emergente'), contar_contratos('Tegucigalpa')
if emergentes: model.Add(sum(emergentes) <= 2)
if tgu: model.Add(sum(tgu) <= 2)

print("⚖️ Restricciones aplicadas (Choques por duración de bloque y bloqueos del docente corregidos).")

⚖️ Restricciones aplicadas (Choques por duración de bloque y bloqueos del docente corregidos).


In [4]:
# ==========================================
# CELDA 4: Función Objetivo (Triage y Demanda Real)
# ==========================================
objetivo = []
for (id_c, s, id_d, patron, h, id_a, mod), var in secciones_posibles.items():
    puntos = 0
    tipo_d = df_docentes[df_docentes['ID_Docente'] == id_d]['Tipo_Docente'].values[0]
    nombre_clase = df_malla[df_malla['ID_Clase'] == id_c]['Nombre_Clase'].values[0].lower()
    
    # Extraemos la cantidad de alumnos proyectados por tu modelo de Machine Learning
    cupos_estimados = df_demanda[df_demanda['ID_Clase'] == id_c]['Cupos_Estimados'].values[0]
    
    # 🛑 1. PRIORIDAD POR DEMANDA (PESO DINÁMICO)
    # A mayor cantidad de alumnos, más puntos. Así aseguramos que se abran las clases más llenas.
    puntos += int(cupos_estimados) * 10
    
    # 🛑 2. ESTRATEGIA DE GRADUACIÓN (Prioridad Absoluta)
    if any(k in nombre_clase for k in ['seminario', 'proyecto', 'práctica', 'practica', 'tesis']):
        puntos += 5000  # Clases Críticas para Egresar
    elif any(k in nombre_clase for k in ['redes', 'sistemas operativos', 'auditoría', 'software']):
        puntos += 1000  # Clases de nivel avanzado 
    else:
        puntos += 100   # Clases generales
        
    # 3. ESTRATEGIA PRESUPUESTARIA
    if tipo_d == 'Base': puntos += 20          
    elif tipo_d == 'Tegucigalpa': puntos += 5           
    else: puntos += 1           
    
    # 4. ESTRATEGIA DE HORARIOS Y AULAS
    if mod == 'Virtual':
        if h >= 17: puntos += 10      
        elif h < 12: puntos -= 5       
    else:
        necesita_lab = df_malla[df_malla['ID_Clase'] == id_c]['Requiere_Laboratorio'].values[0]
        es_lab = 1 if df_aulas[df_aulas['ID_Espacio'] == id_a]['Tipo_Espacio'].values[0].lower() == 'laboratorio' else 0
        if necesita_lab == es_lab:
            puntos += 5 
        
    if patron == 'Lu,Ma,Mi,Ju': puntos += 5 

    objetivo.append(var * puntos)

model.Maximize(sum(objetivo))

solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"✅ ¡Horario generado exitosamente! (Puntuación: {solver.ObjectiveValue()})")
else:
    print("❌ No se encontró solución.")

✅ ¡Horario generado exitosamente! (Puntuación: 31168.0)


In [5]:
# ==========================================
# CELDA 5: Calendario Maestro y Análisis de Brechas
# ==========================================
import pandas as pd

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    horario_flat = []
    secciones_abiertas_por_clase = {} 
    
    for v_key, var in secciones_posibles.items():
        if solver.Value(var) == 1: 
            id_c, s, id_d, patron, h_inicio, id_a, mod = v_key
            
            # 🛑 EXTRAEMOS EL CÓDIGO Y NOMBRE
            codigo_clase = df_malla[df_malla['ID_Clase'] == id_c]['Codigo_Oficial'].values[0]
            clase = df_malla[df_malla['ID_Clase'] == id_c]['Nombre_Clase'].values[0]
            
            docente = df_docentes[df_docentes['ID_Docente'] == id_d]['Nombre'].values[0]
            aula_nombre = 'Virtual' if mod == 'Virtual' else df_aulas[df_aulas['ID_Espacio'] == id_a]['Nombre_Espacio'].values[0]
            duracion = duraciones[v_key]
            
            secciones_abiertas_por_clase[id_c] = secciones_abiertas_por_clase.get(id_c, 0) + 1
            
            dias_asignados = patron.split(',')
            for dia in dias_asignados:
                for hr_bloque in range(h_inicio, h_inicio + duracion):
                    horario_flat.append({
                        'Hora': f"{hr_bloque:02d}:00", 
                        'Dia': dia,
                        # 🛑 AGREGAMOS EL CÓDIGO A LA VISTA
                        'Detalle': f"📚 {codigo_clase} - {clase} (Sec-{s+1})\n👨‍🏫 {docente}\n🏫 {aula_nombre}"
                    })

    df_flat = pd.DataFrame(horario_flat)
    df_calendario = df_flat.groupby(['Hora', 'Dia'])['Detalle'].apply(lambda x: "\n\n".join(x)).reset_index()
    
    matriz_semanal = df_calendario.pivot(index='Hora', columns='Dia', values='Detalle').fillna("---")
    
    orden_dias = ['Lu', 'Ma', 'Mi', 'Ju', 'Vi', 'Sa', 'Do']
    columnas_presentes = [d for d in orden_dias if d in matriz_semanal.columns]
    matriz_semanal = matriz_semanal[columnas_presentes]

    print("🎓 CALENDARIO MAESTRO SEMANAL COMPLETADO:")
    display(matriz_semanal.style.set_properties(**{'white-space': 'pre-wrap', 'vertical-align': 'top', 'border': '1px solid black', 'padding': '10px'}))
    
    # ---------------------------------------------------------
    # RETROALIMENTACIÓN DE GESTIÓN (GAP ANALYSIS)
    # ---------------------------------------------------------
    print("\n" + "="*80)
    print("📊 REPORTE DE GESTIÓN Y SUGERENCIAS DE CONTRATACIÓN (GAP ANALYSIS)")
    print("="*80)
    
    alertas = 0
    for idx, row in df_demanda.iterrows():
        id_c = row['ID_Clase']
        cupos_necesarios = row['Cupos_Estimados']
        secciones_pedidas = int(np.ceil(cupos_necesarios / CAPACIDAD_ESTANDAR))
        
        secciones_logradas = secciones_abiertas_por_clase.get(id_c, 0)
        
        if secciones_pedidas > secciones_logradas:
            alertas += 1
            clase_nombre = row['Nombre_Clase']
            codigo_oficial = df_malla[df_malla['ID_Clase'] == id_c]['Codigo_Oficial'].values[0]
            area_id = df_malla[df_malla['ID_Clase'] == id_c]['ID_Area'].values[0]
            area_nombre = pd.read_sql(f"SELECT Nombre_Area FROM areas_academicas WHERE ID_Area = {area_id}", engine).values[0][0]
            
            print(f"⚠️ DÉFICIT EN: {codigo_oficial} - {clase_nombre}")
            print(f"   - Demanda proyectada: {cupos_necesarios} alumnos ({secciones_pedidas} secciones requeridas).")
            print(f"   - Cobertura lograda:  {secciones_logradas} secciones abiertas en este horario.")
            print(f"   - 💡 SUGERENCIA DIRECTIVA: Para suplir esta demanda, se requiere habilitar disponibilidad ")
            print(f"        o contratar un Ingeniero con especialidad en el Área: '{area_nombre}'.")
            print("-" * 60)
            
    if alertas == 0:
        print("✅ EXCELENTE: La oferta académica actual cubre el 100% de la demanda proyectada sin déficit de personal o aulas.")
else:
    print("❌ No se encontró solución.")

🎓 CALENDARIO MAESTRO SEMANAL COMPLETADO:


Dia,Lu,Ma,Mi,Ju,Vi
Hora,,,,,
07:00,📚 IS-115 - Seminario de Investigación (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-914 - Liderazgo para el Cambio (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111,📚 IS-115 - Seminario de Investigación (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-914 - Liderazgo para el Cambio (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111,📚 IS-115 - Seminario de Investigación (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-914 - Liderazgo para el Cambio (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111,📚 IS-115 - Seminario de Investigación (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-914 - Liderazgo para el Cambio (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111,---
08:00,📚 ISC-204 - Paradigmas de Programación (Sec-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 321 📚 ISC-443 - Industria de TI (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111 📚 IS-412 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323,📚 ISC-204 - Paradigmas de Programación (Sec-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 321 📚 ISC-443 - Industria de TI (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111 📚 IS-412 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323,📚 ISC-204 - Paradigmas de Programación (Sec-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 321 📚 ISC-443 - Industria de TI (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111 📚 IS-412 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323,📚 ISC-204 - Paradigmas de Programación (Sec-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 321 📚 ISC-443 - Industria de TI (Sec-1) 👨‍🏫 Lic. Marisol Cruz / Yeri / Susan 🏫 Aula 111 📚 IS-412 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323,---
09:00,📚 IS-115 - Seminario de Investigación (Sec-2) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-802 - Ingeniería del Software (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 📚 ISC-334 - Sistemas Operativos II (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 404,📚 IS-115 - Seminario de Investigación (Sec-2) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-802 - Ingeniería del Software (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 📚 ISC-334 - Sistemas Operativos II (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 404,📚 IS-115 - Seminario de Investigación (Sec-2) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-802 - Ingeniería del Software (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 📚 ISC-334 - Sistemas Operativos II (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 404,📚 IS-115 - Seminario de Investigación (Sec-2) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 📚 IS-802 - Ingeniería del Software (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 📚 ISC-334 - Sistemas Operativos II (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 404,---
10:00,📚 IS-913 - Diseño de Compiladores (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 403 📚 IS-601 - Base de Datos II (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 404 📚 ISC-333 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Aula 323 📚 ISC-409 - Calidad de Software (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111,📚 IS-913 - Diseño de Compiladores (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 403 📚 IS-601 - Base de Datos II (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 404 📚 ISC-333 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Aula 323 📚 ISC-409 - Calidad de Software (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111,📚 IS-913 - Diseño de Compiladores (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 403 📚 IS-601 - Base de Datos II (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 404 📚 ISC-333 - Sistemas Operativos I (Sec-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Aula 323 📚 ISC-409 - Calidad de Software (Sec-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111,📚 IS-913 - Diseño de Compiladores (Sec-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 403 📚 IS-601 - Base de Datos II (Sec-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 4


📊 REPORTE DE GESTIÓN Y SUGERENCIAS DE CONTRATACIÓN (GAP ANALYSIS)
⚠️ DÉFICIT EN: ISC-312 - Teoría de la Computación
   - Demanda proyectada: 12 alumnos (1 secciones requeridas).
   - Cobertura lograda:  0 secciones abiertas en este horario.
   - 💡 SUGERENCIA DIRECTIVA: Para suplir esta demanda, se requiere habilitar disponibilidad 
        o contratar un Ingeniero con especialidad en el Área: 'Teoría y Metodologías de la Computación'.
------------------------------------------------------------
⚠️ DÉFICIT EN: IE-326 - Instalaciones Eléctricas para Centros de Datos
   - Demanda proyectada: 12 alumnos (1 secciones requeridas).
   - Cobertura lograda:  0 secciones abiertas en este horario.
   - 💡 SUGERENCIA DIRECTIVA: Para suplir esta demanda, se requiere habilitar disponibilidad 
        o contratar un Ingeniero con especialidad en el Área: 'Infraestructura de TI'.
------------------------------------------------------------
⚠️ DÉFICIT EN: IS-603 - Arquitectura de Computadoras
   - Dema